# 18 — Checkpoint Pattern

Purpose: cut long Spark lineage and continue from a materialized intermediate DataFrame.

Process schema:

```text
SOURCE DATA
  |
  v
LONG TRANSFORMATION CHAIN
  |
  v
LOCAL CHECKPOINT
  |
  v
CONTINUE PIPELINE FROM MATERIALIZED DATA
```

For this standalone notebook, use:

```python
df.localCheckpoint(eager=True)
```

Why:

```text
checkpoint()      = reliable checkpoint, needs shared checkpoint storage
localCheckpoint() = executor-local checkpoint, practical for this local Spark notebook
```

In a real cluster, `checkpoint()` should point to shared storage such as HDFS/S3/ADLS or another filesystem available to all Spark workers.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("checkpoint-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Input data

In [2]:
schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_type", StringType(), False),
])

rows = [
    (f"e{i}", f"u{i % 3}", float(i), "purchase" if i % 4 != 0 else "refund")
    for i in range(1, 31)
]

events = spark.createDataFrame(rows, schema)

events.show(10, truncate=False)
print("rows:", events.count())

+--------+-------+------+----------+
|event_id|user_id|amount|event_type|
+--------+-------+------+----------+
|e1      |u1     |1.0   |purchase  |
|e2      |u2     |2.0   |purchase  |
|e3      |u0     |3.0   |purchase  |
|e4      |u1     |4.0   |refund    |
|e5      |u2     |5.0   |purchase  |
|e6      |u0     |6.0   |purchase  |
|e7      |u1     |7.0   |purchase  |
|e8      |u2     |8.0   |refund    |
|e9      |u0     |9.0   |purchase  |
|e10     |u1     |10.0  |purchase  |
+--------+-------+------+----------+
only showing top 10 rows
rows: 30


## Step 2 — Build longer lineage

In [3]:
prepared = events

for i in range(1, 8):
    prepared = (
        prepared
        .withColumn(f"amount_step_{i}", F.col("amount") + F.lit(float(i)))
        .withColumn(f"is_large_step_{i}", F.col(f"amount_step_{i}") > F.lit(10.0 + i))
    )

prepared = (
    prepared
    .withColumn(
        "signed_amount",
        F.when(F.col("event_type") == "refund", -F.col("amount")).otherwise(F.col("amount"))
    )
)

prepared.show(5, truncate=False)

+--------+-------+------+----------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+
|event_id|user_id|amount|event_type|amount_step_1|is_large_step_1|amount_step_2|is_large_step_2|amount_step_3|is_large_step_3|amount_step_4|is_large_step_4|amount_step_5|is_large_step_5|amount_step_6|is_large_step_6|amount_step_7|is_large_step_7|signed_amount|
+--------+-------+------+----------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+
|e1      |u1     |1.0   |purchase  |2.0          |false          |3.0          |false          |4.0          |false          |5.0          |false          |6.0          |false          |7.0          |false          |8

## Step 3 — Physical plan before local checkpoint

In [4]:
prepared.explain()

== Physical Plan ==
*(1) Project [event_id#0, user_id#1, amount#2, event_type#3, amount_step_1#25, is_large_step_1#26, amount_step_2#27, is_large_step_2#28, amount_step_3#29, is_large_step_3#30, amount_step_4#31, is_large_step_4#32, amount_step_5#33, is_large_step_5#34, amount_step_6#35, is_large_step_6#36, amount_step_7#37, (amount_step_7#37 > 17.0) AS is_large_step_7#38, CASE WHEN (event_type#3 = refund) THEN -amount#2 ELSE amount#2 END AS signed_amount#39]
+- *(1) Project [event_id#0, user_id#1, amount#2, event_type#3, amount_step_1#25, is_large_step_1#26, amount_step_2#27, is_large_step_2#28, amount_step_3#29, is_large_step_3#30, amount_step_4#31, is_large_step_4#32, amount_step_5#33, is_large_step_5#34, amount_step_6#35, (amount_step_6#35 > 16.0) AS is_large_step_6#36, (amount#2 + 7.0) AS amount_step_7#37]
   +- *(1) Project [event_id#0, user_id#1, amount#2, event_type#3, amount_step_1#25, is_large_step_1#26, amount_step_2#27, is_large_step_2#28, amount_step_3#29, is_large_step_3#

## Step 4 — Local checkpoint the intermediate DataFrame

This is the notebook-safe version:

```python
prepared_checkpointed = prepared.localCheckpoint(eager=True)
```

It truncates lineage without requiring a shared reliable checkpoint directory.

In [5]:
prepared_checkpointed = prepared.localCheckpoint(eager=True)

prepared_checkpointed.count()
prepared_checkpointed.show(5, truncate=False)

+--------+-------+------+----------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+
|event_id|user_id|amount|event_type|amount_step_1|is_large_step_1|amount_step_2|is_large_step_2|amount_step_3|is_large_step_3|amount_step_4|is_large_step_4|amount_step_5|is_large_step_5|amount_step_6|is_large_step_6|amount_step_7|is_large_step_7|signed_amount|
+--------+-------+------+----------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+---------------+-------------+
|e1      |u1     |1.0   |purchase  |2.0          |false          |3.0          |false          |4.0          |false          |5.0          |false          |6.0          |false          |7.0          |false          |8

## Step 5 — Physical plan after local checkpoint

In [6]:
prepared_checkpointed.explain()

== Physical Plan ==
*(1) Scan ExistingRDD[event_id#0,user_id#1,amount#2,event_type#3,amount_step_1#25,is_large_step_1#26,amount_step_2#27,is_large_step_2#28,amount_step_3#29,is_large_step_3#30,amount_step_4#31,is_large_step_4#32,amount_step_5#33,is_large_step_5#34,amount_step_6#35,is_large_step_6#36,amount_step_7#37,is_large_step_7#38,signed_amount#39]




## Step 6 — Continue pipeline after checkpoint

In [7]:
gold_by_user = (
    prepared_checkpointed
    .groupBy("user_id")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("signed_amount").alias("net_amount"),
        F.avg("amount").alias("avg_amount"),
    )
    .orderBy("user_id")
)

gold_by_user.show(truncate=False)

+-------+-----------+----------+----------+
|user_id|event_count|net_amount|avg_amount|
+-------+-----------+----------+----------+
|u0     |10         |93.0      |16.5      |
|u1     |10         |49.0      |14.5      |
|u2     |10         |99.0      |15.5      |
+-------+-----------+----------+----------+



## Step 7 — Reliable checkpoint variant

Use this only when the checkpoint path is on shared storage visible to all Spark nodes.

```python
spark.sparkContext.setCheckpointDir("hdfs:///tmp/checkpoints")
prepared_checkpointed = prepared.checkpoint(eager=True)
```

For the current Docker standalone notebook, `localCheckpoint()` is the safer example.

## Step 8 — Cache vs local checkpoint

In [8]:
cached = prepared.cache()
cached.count()

print("cached storage level:", cached.storageLevel)

cached.unpersist()

cached storage level: Disk Memory Deserialized 1x Replicated


DataFrame[event_id: string, user_id: string, amount: double, event_type: string, amount_step_1: double, is_large_step_1: boolean, amount_step_2: double, is_large_step_2: boolean, amount_step_3: double, is_large_step_3: boolean, amount_step_4: double, is_large_step_4: boolean, amount_step_5: double, is_large_step_5: boolean, amount_step_6: double, is_large_step_6: boolean, amount_step_7: double, is_large_step_7: boolean, signed_amount: double]

## Step 9 — Control totals

In [9]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("input_rows", events.count()),
        ("prepared_rows", prepared.count()),
        ("checkpointed_rows", prepared_checkpointed.count()),
        ("gold_rows", gold_by_user.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+-----------------+-----+
|metric           |value|
+-----------------+-----+
|input_rows       |30   |
|prepared_rows    |30   |
|checkpointed_rows|30   |
|gold_rows        |3    |
+-----------------+-----+



## Final result

```text
LONG LINEAGE
  |
  v
LOCAL CHECKPOINT
  |
  v
SHORTER DOWNSTREAM PLAN
```

In [10]:
spark.stop()